In [5]:
"""
Rekonstruiert den LST-/NDVI-Verlauf (NHDA vs. RA) inkl. Differenz
für eine einzelne NHDA aus dem bereits kombinierten GPKG.

Erwartete Spalten im COMBINED_GPKG (aus den LST_/NDVI_Comparison-Skripten):
    nhda_median_LST_<jahr>,  ra_median_LST_<jahr>,  difference_LST_<jahr>,
    nhda_std_LST_<jahr>,     ra_std_LST_<jahr>
    nhda_median_NDVI_<jahr>, ra_median_NDVI_<jahr>, difference_NDVI_<jahr>,
    nhda_std_NDVI_<jahr>,    ra_std_NDVI_<jahr>
    nhda_id, type ('NHDA' oder 'RA' -> nur die Geometrie unterscheidet sich,
    die Wertespalten sind für beide Zeilen identisch)
"""

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os

# ============================================================================
# KONFIGURATION
# ============================================================================
TARGET_NHDA_ID = "09186_5"

COMBINED_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI.gpkg"
CONSTRUCTION_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"

OUTPUT_DIR = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Single_NHDA_Plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONSTRUCTION_SPECIAL_MAP = {
    'AUC_2015': 2014,
    'AUC_2016': 2015,
}

# ============================================================================
# HILFSFUNKTIONEN
# ============================================================================
def year_columns(frame, prefix):
    """Findet Spalten der Form <prefix><jahr>, z.B. nhda_median_LST_2019."""
    pattern = re.compile(rf"^{re.escape(prefix)}(\d{{4}})$")
    result = {}
    for col in frame.columns:
        m = pattern.match(col)
        if m:
            result[int(m.group(1))] = col
    return result


def build_long(row, value_cols, std_cols=None):
    """Baut aus einer Zeile eine long-Tabelle jahr -> wert (+ std, falls vorhanden)."""
    records = []
    for year, col in sorted(value_cols.items()):
        value = row.get(col, np.nan)
        std_val = np.nan
        if std_cols and year in std_cols:
            std_val = row.get(std_cols[year], np.nan)
        records.append({'year': year, 'value': value, 'std': std_val})
    df = pd.DataFrame(records)
    return df.dropna(subset=['value'])


# ============================================================================
# 1. DATEN LADEN
# ============================================================================
print("=" * 80)
print(f"EINZEL-NHDA VERLAUF: {TARGET_NHDA_ID}")
print("=" * 80)

for path in [COMBINED_GPKG, CONSTRUCTION_GPKG]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")

gdf = gpd.read_file(COMBINED_GPKG)
gdf['nhda_id'] = gdf['nhda_id'].astype(str)

subset = gdf[gdf['nhda_id'] == TARGET_NHDA_ID].copy()
if subset.empty:
    raise ValueError(f"NHDA-ID {TARGET_NHDA_ID} nicht im Combined-GPKG gefunden.")

type_col = None
for candidate in ['type', 'area_type']:
    if candidate in subset.columns:
        type_col = candidate
        break
if type_col is None:
    raise ValueError("Keine Typ-Spalte ('type' oder 'area_type') gefunden.")

print(f"   Gefundene Zeilen: {len(subset)} (Typen: {subset[type_col].unique().tolist()})")

# Werte-Zeile: Wertespalten sind für NHDA- und RA-Zeile identisch, wir nehmen die NHDA-Zeile
value_row = subset[subset[type_col].astype(str) == 'NHDA']
if value_row.empty:
    value_row = subset.iloc[[0]]
value_row = value_row.iloc[0]

# ============================================================================
# 2. CONSTRUCTION START
# ============================================================================
gdf_const = gpd.read_file(CONSTRUCTION_GPKG)
id_candidates = ['nhda_id', 'cluster_id', 'nda_id']
id_col = next((c for c in id_candidates if c in gdf_const.columns), None)
if id_col is None:
    raise ValueError(f"Keine ID-Spalte im Construction-GPKG gefunden. Versucht: {id_candidates}")
if 'construction_start_year' not in gdf_const.columns:
    raise ValueError("Spalte 'construction_start_year' nicht im Construction-GPKG gefunden.")

gdf_const['nhda_id'] = gdf_const[id_col].astype(str)
const_row = gdf_const[gdf_const['nhda_id'] == TARGET_NHDA_ID]
if const_row.empty:
    raise ValueError(f"NHDA-ID {TARGET_NHDA_ID} nicht im Construction-GPKG gefunden.")

raw_construction = str(const_row.iloc[0]['construction_start_year']).strip()
construction_year = CONSTRUCTION_SPECIAL_MAP.get(raw_construction, raw_construction)
construction_year = float(construction_year)

print(f"   Construction start (roh):       {raw_construction}")
print(f"   Construction start (verwendet): {construction_year}")

# ============================================================================
# 3. SPALTEN FINDEN UND LONG-TABELLEN BAUEN
# ============================================================================
lst_nhda_cols = year_columns(gdf, 'nhda_median_LST_')
lst_ra_cols   = year_columns(gdf, 'ra_median_LST_')
lst_diff_cols = year_columns(gdf, 'difference_LST_')
lst_nhda_std  = year_columns(gdf, 'nhda_std_LST_')
lst_ra_std    = year_columns(gdf, 'ra_std_LST_')

ndvi_nhda_cols = year_columns(gdf, 'nhda_median_NDVI_')
ndvi_ra_cols   = year_columns(gdf, 'ra_median_NDVI_')
ndvi_diff_cols = year_columns(gdf, 'difference_NDVI_')
ndvi_nhda_std  = year_columns(gdf, 'nhda_std_NDVI_')
ndvi_ra_std    = year_columns(gdf, 'ra_std_NDVI_')

print(f"   LST-Jahre gefunden:  {sorted(lst_nhda_cols.keys())}")
print(f"   NDVI-Jahre gefunden: {sorted(ndvi_nhda_cols.keys())}")

lst_nhda = build_long(value_row, lst_nhda_cols, lst_nhda_std)
lst_ra   = build_long(value_row, lst_ra_cols, lst_ra_std)
lst_diff = build_long(value_row, lst_diff_cols)

ndvi_nhda = build_long(value_row, ndvi_nhda_cols, ndvi_nhda_std)
ndvi_ra   = build_long(value_row, ndvi_ra_cols, ndvi_ra_std)
ndvi_diff = build_long(value_row, ndvi_diff_cols)


# Unsicherheitsband fuer die Differenz: da keine eigene diff-std-Spalte existiert,
# wird die Standardabweichung der Differenz über Fehlerfortpflanzung angenähert:
# std_diff = sqrt(std_nhda^2 + std_ra^2)
def propagate_diff_std(diff_df, nhda_df, ra_df):
    if diff_df.empty:
        return diff_df
    merged = diff_df.merge(nhda_df[['year', 'std']], on='year', how='left')
    merged = merged.merge(ra_df[['year', 'std']], on='year', how='left', suffixes=('_nhda', '_ra'))
    merged['std'] = np.sqrt(merged['std_nhda'].fillna(0) ** 2 + merged['std_ra'].fillna(0) ** 2)
    return merged[['year', 'value', 'std']]


lst_diff = propagate_diff_std(lst_diff, lst_nhda, lst_ra)
ndvi_diff = propagate_diff_std(ndvi_diff, ndvi_nhda, ndvi_ra)

# ============================================================================
# 4. PLOT
# ============================================================================
place_col = next((c for c in ['gemeinde', 'kommune', 'ort', 'place_name', 'location'] if c in subset.columns), None)
place_name = value_row[place_col] if place_col else None
title_place = f" in {place_name}" if place_name else ""

fig, axes = plt.subplots(4, 1, figsize=(9, 16))
fig.suptitle(f"New Housing Development Area ({TARGET_NHDA_ID}){title_place}", fontweight='bold', fontsize=13)


def plot_two_lines(ax, series_a, series_b, label_a, label_b, color_a, color_b, ylabel, title):
    for series, label, color in [(series_a, label_a, color_a), (series_b, label_b, color_b)]:
        if series.empty:
            continue
        ax.plot(series['year'], series['value'], marker='o', color=color, label=label, linewidth=2)
        if series['std'].notna().any():
            ax.fill_between(
                series['year'],
                series['value'] - series['std'],
                series['value'] + series['std'],
                color=color, alpha=0.2
            )
    ax.axvline(construction_year, linestyle='--', color='olive', linewidth=1.8, label='Construction start')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Year')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend()


def plot_diff(ax, series, color, ylabel, title, diff_label):
    if not series.empty:
        ax.plot(series['year'], series['value'], marker='o', color=color, linewidth=2, label=diff_label)
        if series['std'].notna().any():
            ax.fill_between(
                series['year'],
                series['value'] - series['std'],
                series['value'] + series['std'],
                color=color, alpha=0.2
            )
    ax.axhline(0, linestyle='--', color='red', linewidth=1.5)
    ax.axvline(construction_year, linestyle='--', color='olive', linewidth=1.8, label='Construction start')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Year')
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend()


plot_two_lines(axes[0], lst_nhda, lst_ra, 'NHDA', 'RA', 'red', 'blue', 'LST [°C]', 'LST')
plot_diff(axes[1], lst_diff, 'darkred', 'ΔLST [°C]', 'ΔLST', 'Difference (LST)')
plot_two_lines(axes[2], ndvi_nhda, ndvi_ra, 'NHDA', 'RA', 'red', 'blue', 'NDVI', 'NDVI')
plot_diff(axes[3], ndvi_diff, 'darkgreen', 'ΔNDVI', 'ΔNDVI', 'Difference (NDVI)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = f"{OUTPUT_DIR}/{TARGET_NHDA_ID}_trajectory.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n   Plot gespeichert: {out_path}")
print("\nFERTIG")

EINZEL-NHDA VERLAUF: 09186_5
   Gefundene Zeilen: 2 (Typen: ['NHDA', 'RA'])
   Construction start (roh):       2020
   Construction start (verwendet): 2020.0
   LST-Jahre gefunden:  [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
   NDVI-Jahre gefunden: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


KeyError: 'std_nhda'